# LunaVision AI — Kaggle GPU pipeline
Preprocess → feature extraction → classifier training.
**Before running:** Accelerator = GPU, Internet = On, and attach your data as a Kaggle Dataset (see `DATA_INPUT` below).

In [ ]:
# 1. GPU check
!nvidia-smi -L
import torch
print("torch cuda:", torch.cuda.is_available(), torch.cuda.device_count())

In [ ]:
# 2. Get the code
import os
REPO = "https://github.com/SandeepCodez24/LunaVisionAi.git"
# Private repo? Add a Kaggle Secret named GITHUB_TOKEN and uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO = f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@github.com/SandeepCodez24/LunaVisionAi.git"

if not os.path.exists("/kaggle/working/LunaVisionAi"):
    !git clone --depth 1 {REPO} /kaggle/working/LunaVisionAi
%cd /kaggle/working/LunaVisionAi/exoplanet_pipeline

In [ ]:
# 3. Dependencies (torch, sklearn, xgboost, lightgbm, optuna, shap ship with Kaggle)
!pip install -q lightkurve transitleastsquares wotan astropy tsfresh imbalanced-learn cupy-cuda12x psutil tqdm

In [ ]:
# 4. Data: copy from the attached Kaggle Dataset into the writable working dir.
# Adjust DATA_INPUT to your dataset slug. Expected layout inside it:
#   catalogs/            (CSV files)
#   lc_raw/              (*.npz)
#   lc_detrended/        (optional, skips preprocessing)
#   lc_injected/         (optional)
import shutil, glob
DATA_INPUT = "/kaggle/input/lunavision-data"
print(os.listdir(DATA_INPUT))

os.makedirs("data/processed", exist_ok=True)
for src, dst in [("catalogs", "data/catalogs"),
                 ("lc_raw", "data/processed/lc_raw"),
                 ("lc_detrended", "data/processed/lc_detrended"),
                 ("lc_injected", "data/processed/lc_injected")]:
    p = os.path.join(DATA_INPUT, src)
    if os.path.isdir(p) and not os.path.exists(dst):
        shutil.copytree(p, dst); print("copied", src)
for d in ["lc_raw", "lc_detrended", "lc_injected"]:
    print(d, len(glob.glob(f"data/processed/{d}/*.npz")))

In [ ]:
# 5. Stage: preprocessing (skip if lc_detrended already populated)
if len(glob.glob("data/processed/lc_detrended/*.npz")) == 0:
    !python src/preprocessing.py --n-jobs -1
else:
    print("lc_detrended present, skipping")

In [ ]:
# 6. Stage: feature extraction (GPU). Resumable via checkpoints; add --no-resume to restart.
!python src/features.py

In [ ]:
# 7. Stage: train classifier
!python src/classifier.py --n-jobs 4

In [ ]:
# 8. Save results: copy to /kaggle/working/output (kept when you Save Version -> Save & Run All)
os.makedirs("/kaggle/working/output", exist_ok=True)
for f in ["models/classifier.joblib", "models/metrics.json", "data/catalogs/feature_matrix.csv"]:
    if os.path.exists(f): shutil.copy(f, "/kaggle/working/output/")
!ls -lh /kaggle/working/output
# Optional: free the big data copies so the output stays small
# shutil.rmtree("data/processed", ignore_errors=True)